# Building a Neural Network from the Ground Up

This notebook walks through a complete implementation of a feedforward neural network built entirely with NumPy. Rather than using deep learning libraries like TensorFlow or PyTorch, we manually code each component — **forward propagation**, **activation functions**, **backpropagation**, and **gradient descent** — to gain a deeper understanding of what happens under the hood.

The network is trained on a handwritten digit classification task and evaluated on held-out test data using accuracy as the primary metric.

In [ ]:
import numpy as np
from tensorflow import keras

## Setting Up Weights and Biases (Random Initialization)

We initialize all weights and biases using **random values** drawn from a standard normal distribution.

For every layer $l$:
- $W^{[l]} \sim \mathcal{N}(0, 1)$ — weight matrix sampled from a Gaussian
- $b^{[l]}$ initialized to zeros

Random initialization breaks symmetry between neurons so that each unit can learn distinct features during training. Biases are set to zero since the random weights already provide asymmetry.

In [ ]:
def init_network(arch):
    np.random.seed(7)
    net_params = {}
    for l in range(1, len(arch)):
        net_params['W' + str(l)] = np.random.randn(arch[l], arch[l - 1]) * 0.01
        net_params['b' + str(l)] = np.zeros((arch[l], 1))
    return net_params

## Defining Activation Functions

Two activation functions are used in this network:

**ReLU** — applied at every hidden layer, it introduces non-linearity by zeroing out negative values:

$$\text{ReLU}(z) = \max(0, z)$$

Its gradient is straightforward:

$$\text{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & \text{otherwise} \end{cases}$$

**Softmax** — used at the output layer to convert raw scores into a probability distribution across classes:

$$\sigma(z_i) = \frac{e^{z_i}}{\sum_{k} e^{z_k}}$$


In [ ]:
def activation_relu(z):
    return np.maximum(0, z)

def drelu(z):
    return (z > 0).astype(float)

def activation_softmax(z):
    shifted = np.exp(z - np.max(z, axis=0, keepdims=True))
    return shifted / np.sum(shifted, axis=0, keepdims=True)

## Forward Pass

Data flows through the network layer by layer. At each layer $l$, we compute:

$$Z^{[l]} = W^{[l]} \cdot A^{[l-1]} + b^{[l]}$$

$$A^{[l]} = g(Z^{[l]})$$

Here $A^{[0]} = X$ is the raw input. The function $g$ is ReLU for all hidden layers, and Softmax for the final output layer. We store all intermediate values for use during backpropagation.


In [ ]:
def forward_pass(X, arch, net_params):
    stored = {'A0': X}
    act = X

    for l in range(1, len(arch)):
        w = net_params['W' + str(l)]
        b = net_params['b' + str(l)]
        z = np.dot(w, act) + b

        if l < len(arch) - 1:
            act = activation_relu(z)
        else:
            act = activation_softmax(z)

        stored['Z' + str(l)] = z
        stored['A' + str(l)] = act

    return act, stored

## Computing the Loss (Cross-Entropy)

To measure how far the network's predictions are from the true labels, we use the **categorical cross-entropy** loss:

$$\mathcal{L} = -\frac{1}{m} \sum_{i=1}^{m} \sum_{c=1}^{C} y_c^{(i)} \ln(\hat{y}_c^{(i)})$$

- $m$ — total number of training samples
- $C$ — number of output classes
- $y$ — one-hot encoded ground truth
- $\hat{y}$ — predicted probabilities from the softmax layer


In [ ]:
def categorical_cross_entropy(output, Y):
    n = Y.shape[1]
    cost = -np.sum(Y * np.log(output + 1e-8)) / n
    return cost

## Backward Pass (Backpropagation)

Gradients are propagated backward through the network using the **chain rule**. For each layer $l$:

**Error signal:**

$$\delta^{[l]} = (W^{[l+1]})^T \delta^{[l+1]} \circ g'(Z^{[l]})$$

**Gradients for weights and biases:**

$$\frac{\partial \mathcal{L}}{\partial W^{[l]}} = \frac{1}{m} \delta^{[l]} (A^{[l-1]})^T \qquad \frac{\partial \mathcal{L}}{\partial b^{[l]}} = \frac{1}{m} \sum \delta^{[l]}$$

**Parameter update via gradient descent:**

$$W^{[l]} \leftarrow W^{[l]} - \alpha \cdot \nabla_{W} \mathcal{L}$$


In [ ]:
def backward_pass(stored, net_params, Y, arch, lr=0.01):
    n = Y.shape[1]
    depth = len(arch) - 1

    dz = stored['A' + str(depth)] - Y

    for l in reversed(range(1, depth + 1)):
        a_prev = stored['A' + str(l - 1)]
        z_val = stored['Z' + str(l)]
        w = net_params['W' + str(l)]

        if l < depth:
            dz = dz * drelu(z_val)

        dw = np.dot(dz, a_prev.T) / n
        db = np.sum(dz, axis=1, keepdims=True) / n

        net_params['W' + str(l)] -= lr * dw
        net_params['b' + str(l)] -= lr * db

        if l > 1:
            dz = np.dot(w.T, dz)

## Generating Predictions

To classify an input, we pick the neuron with the highest probability from the softmax output:

$$\hat{y} = \underset{c}{\arg\max} \; \sigma(Z^{[L]})_c$$


In [ ]:
def classify(output):
    return np.argmax(output, axis=0)

def accuracy_score(pred, Y):
    true_labels = np.argmax(Y, axis=0)
    return np.mean(pred == true_labels)

## Training Loop

Each iteration of the training loop performs the following steps:
1. Run the forward pass to get predictions
2. Calculate the cross-entropy loss
3. Execute the backward pass to compute gradients and update parameters
4. Periodically log the loss and accuracy to monitor progress

In [ ]:
def fit_model(X, Y, arch, epochs=1000, lr=0.01):
    net_params = init_network(arch)

    for epoch in range(epochs):
        output, stored = forward_pass(X, arch, net_params)
        cost = categorical_cross_entropy(output, Y)
        backward_pass(stored, net_params, Y, arch, lr)

        if epoch % 100 == 0:
            pred = classify(output)
            acc = accuracy_score(pred, Y)
            print(f"Epoch {epoch}: Loss = {cost:.4f} | Accuracy = {acc:.4f}")

    return net_params

In [ ]:
def inference(X, net_params, arch):
    output, _ = forward_pass(X, arch, net_params)
    return np.argmax(output, axis=0)

## Loading and Preparing the MNIST Dataset

The MNIST dataset consists of 70,000 grayscale images of handwritten digits (0 through 9), each sized 28x28 pixels.

**Preprocessing pipeline:**
1. Fetch the dataset via `keras.datasets.mnist`
2. Scale pixel intensities to $[0, 1]$ by dividing by 255
3. Flatten each 28x28 image into a 784-dimensional vector and transpose so columns represent samples
4. One-hot encode the labels — e.g., digit `5` becomes `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`

In [ ]:
(train_imgs, train_y), (test_imgs, test_y) = keras.datasets.mnist.load_data()

train_X = train_imgs.reshape(-1, 784).T / 255.0
test_X = test_imgs.reshape(-1, 784).T / 255.0

train_labels = np.eye(10)[train_y].T
test_labels = np.eye(10)[test_y].T

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
arch = [784, 64, 32, 10]
model_weights = fit_model(train_X, train_labels, arch, epochs=800, lr=0.1)

Iteration 0: Loss = 3.1170 | Accuracy = 0.0993
Iteration 100: Loss = 0.5321 | Accuracy = 0.8602
Iteration 200: Loss = 0.3786 | Accuracy = 0.8938
Iteration 300: Loss = 0.3333 | Accuracy = 0.9042
Iteration 400: Loss = 0.3089 | Accuracy = 0.9111
Iteration 500: Loss = 0.2916 | Accuracy = 0.9157
Iteration 600: Loss = 0.2776 | Accuracy = 0.9198
Iteration 700: Loss = 0.2649 | Accuracy = 0.9235


In [ ]:
predictions = inference(test_X, model_weights, arch)
ground_truth = np.argmax(test_labels, axis=0)

test_acc = np.mean(predictions == ground_truth) * 100
print(f"Test Accuracy: {test_acc:.2f}%")

Test Accuracy: 92.88%


In [ ]:
model_weights

{'W1': array([[ 0.01773979, -0.00493801,  0.02313173, ..., -0.04769087,
          0.01357849,  0.02180663],
        [ 0.01999252,  0.03859931,  0.02978293, ..., -0.01004741,
          0.00239253,  0.0184264 ],
        [-0.05580521, -0.01889474,  0.0283666 , ..., -0.01400045,
          0.03785487,  0.02203593],
        ...,
        [ 0.03201384,  0.02675996, -0.01336246, ...,  0.02256244,
         -0.02670094,  0.04678863],
        [-0.02087065,  0.03962292,  0.08090054, ...,  0.02943682,
          0.00890256, -0.00427599],
        [ 0.00222667,  0.00266519,  0.04197013, ..., -0.02032842,
          0.05708488, -0.0234534 ]]),
 'b1': array([[ 0.9625342 ],
        [ 0.07774612],
        [ 0.7770016 ],
        [-0.81778456],
        [-0.27118468],
        [-0.02313028],
        [-0.63129848],
        [ 1.24673499],
        [-0.76428007],
        [-0.56941276],
        [ 1.40858072],
        [ 1.15885431],
        [ 1.58367162],
        [-1.90827316],
        [ 0.25169692],
        [ 0.1272